In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def standardize_grid_event_columns(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("region", F.upper(F.trim(F.col("region"))))
        .withColumn("event_type", F.upper(F.trim(F.col("event_type"))))
        .withColumn("severity", F.upper(F.trim(F.col("severity"))))
    )

def cast_grid_event_fields(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("duration_minutes", F.col("duration_minutes").cast("int"))
        .withColumn("event_timestamp", F.to_timestamp("event_ts"))
    )

def add_grid_event_day(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("event_day", F.to_date("event_timestamp"))
    )

def filter_invalid_grid_events(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(F.col("duration_minutes").isNotNull())
        .filter(F.col("duration_minutes") >= 0)
        .filter(F.col("event_day").isNotNull())
    )
def transform_grid_events(df: DataFrame) -> DataFrame:
    """Complete grid events transformation pipeline"""
    return (
        df
        .transform(standardize_grid_event_columns)
        .transform(cast_grid_event_fields)
        .transform(add_grid_event_day)
        .transform(filter_invalid_grid_events)
    )

In [0]:

import yaml

config_path = "/Workspace/Repos/adb-emily@startsteps.org/vattenfall-week9-capstone-EmilyImunde/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
bronze_schema = config["schemas"]["raw"]
silver_schema = config["schemas"]["refined"]

print(f"Catalog: {catalog_name}")
print(f"Bronze Schema: {bronze_schema}")
print(f"Silver Schema: {silver_schema}")

In [0]:
bronze_df = spark.table(f"{catalog_name}.{bronze_schema}.bronze_grid_events")
silver_df = transform_grid_events(bronze_df)
silver_df.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{silver_schema}.silver_grid_events"
)

print(f"✓ Successfully created {catalog_name}.{silver_schema}.silver_grid_events")
print(f"Total rows: {silver_df.count()}")

In [0]:
silver_table = spark.table("vattenfall_dev.refined.silver_grid_events")

print(f"Total rows: {silver_table.count()}")
print("\nSchema:")
silver_table.printSchema()
print("\nFirst 10 rows:")
display(silver_table.limit(10))

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM vattenfall_dev.refined.silver_grid_events;